# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My Baseline Rule

The rule is intentionally simple and transparent.

Content with higher Google Search Console clicks and impressions, together with better search position, receives a higher score.

Reason Codes

- HIGH_PERFORMER: High clicks and good search position.
- GROWING_CONTENT: Moderate performance with growth potential.
- LOW_VISIBILITY: Low clicks and poor search position.
- CTR_OPPORTUNITY: High impressions but relatively low clicks.

Action Labels

- Protect
- Monitor
- Improve SEO
- Rewrite

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

In [4]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [5]:
print("REL exists:", "REL" in globals())
print("con exists:", "con" in globals())

REL exists: True
con exists: True


In [6]:
query = f"""
SELECT
    CASE
        WHEN gsc_clicks = 0 THEN 'No Click'
        WHEN gsc_clicks BETWEEN 1 AND 10 THEN 'Low'
        WHEN gsc_clicks BETWEEN 11 AND 50 THEN 'Medium'
        ELSE 'High'
    END AS click_bucket,

    COUNT(*) AS n,
    ROUND(AVG(gsc_impressions),2) AS avg_impressions,
    ROUND(AVG(gsc_sum_position),2) AS avg_position

FROM {REL}

WHERE
    gsc_data_available IS TRUE

GROUP BY click_bucket

ORDER BY
CASE click_bucket
    WHEN 'No Click' THEN 1
    WHEN 'Low' THEN 2
    WHEN 'Medium' THEN 3
    WHEN 'High' THEN 4
END
"""

signal1 = con.sql(query).df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,click_bucket,n,avg_impressions,avg_position
0,No Click,3193080,46.14,639.87
1,Low,412711,293.60,2762.89
2,Medium,5087,2077.79,11167.37
3,High,183,8728.06,33797.90


In [7]:
con.sql(f"""
SELECT *
FROM {REL}
LIMIT 1
""").df().columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [8]:
query = f"""
SELECT
CASE
    WHEN gsc_clicks = 0 THEN 'No Click'
    WHEN gsc_clicks BETWEEN 1 AND 10 THEN 'Low'
    WHEN gsc_clicks BETWEEN 11 AND 50 THEN 'Medium'
    ELSE 'High'
END AS click_bucket,

COUNT(*) AS n,

ROUND(AVG(gsc_impressions),2) AS avg_impressions,

ROUND(AVG(gsc_avg_position),2) AS avg_position

FROM {REL}

WHERE
gsc_data_available IS TRUE

GROUP BY click_bucket

ORDER BY
CASE click_bucket
WHEN 'No Click' THEN 1
WHEN 'Low' THEN 2
WHEN 'Medium' THEN 3
WHEN 'High' THEN 4
END
"""

signal1 = con.sql(query).df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,click_bucket,n,avg_impressions,avg_position
0,No Click,3193080,46.14,16.75
1,Low,412711,293.60,8.79
2,Medium,5087,2077.79,5.20
3,High,183,8728.06,4.10


## Signal Check 1

**Signal:** Google Search Console Clicks

**Verdict:** **CONFIRMED**

The results show a clear relationship between clicks, impressions, and search position. Content with higher click counts also receives higher impressions and better average search positions. Therefore, Google Search Console clicks are a useful signal for identifying different content performance archetypes and can be safely used in the baseline rule.

In [9]:
query = f"""
SELECT
CASE
    WHEN gsc_avg_position <= 5 THEN 'Top 5'
    WHEN gsc_avg_position <= 10 THEN 'Top 10'
    WHEN gsc_avg_position <= 20 THEN 'Top 20'
    ELSE 'Beyond 20'
END AS position_bucket,

COUNT(*) AS n,

ROUND(AVG(gsc_clicks),2) AS avg_clicks,

ROUND(AVG(gsc_impressions),2) AS avg_impressions

FROM {REL}

WHERE gsc_data_available IS TRUE

GROUP BY position_bucket

ORDER BY
CASE position_bucket
WHEN 'Top 5' THEN 1
WHEN 'Top 10' THEN 2
WHEN 'Top 20' THEN 3
WHEN 'Beyond 20' THEN 4
END;
"""

signal2 = con.sql(query).df()

signal2

,position_bucket,n,avg_clicks,avg_impressions
0,Top 5,1263125,0.35,96.51
1,Top 10,920359,0.22,76.01
2,Top 20,519223,0.18,56.60
3,Beyond 20,908354,0.09,65.41


## Signal Check 2

**Signal:** Google Search Console Average Position

**Verdict:** **CONFIRMED**

The results indicate that better search positions are generally associated with higher clicks and stronger visibility. Content ranked in the Top 5 has the highest average clicks (0.35) and impressions (96.51), while clicks gradually decrease as the average position becomes worse (Top 10 → Top 20 → Beyond 20). Although the "Beyond 20" bucket shows slightly higher average impressions than the "Top 20" bucket, it still receives substantially fewer clicks, suggesting lower search effectiveness. Therefore, average search position is confirmed as a reliable signal for identifying different content performance archetypes and is suitable for the baseline scoring rule.

# 2. Build the Ranked Queue

## Baseline Rule

A content item is considered a stronger content archetype when it receives high search visibility, generates more clicks, and maintains a better average search position.

The baseline score combines these three signals to rank content into meaningful performance archetypes.

Each ranked content item receives:
- A baseline score
- A reason code
- An action label

In [10]:
baseline_query = f"""
WITH content_level AS (
    SELECT
        content_hash_id,

        AVG(gsc_impressions) AS avg_impressions,
        AVG(gsc_clicks) AS avg_clicks,
        AVG(gsc_avg_position) AS avg_position,

        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,

        COUNT(*) AS days_observed

    FROM {REL}

    WHERE gsc_data_available IS TRUE

    GROUP BY content_hash_id

    HAVING COUNT(*) >= 3
)

SELECT
    content_hash_id,
    avg_impressions,
    avg_clicks,
    avg_position,
    total_impressions,
    total_clicks,
    days_observed,

    (
        (avg_impressions * 0.01)
        + (avg_clicks * 5)
        + ((21 - LEAST(avg_position, 20)) * 3)
    ) AS baseline_score,

    CASE
        WHEN avg_clicks >= 10
             AND avg_position <= 10
            THEN 'HIGH_PERFORMANCE'

        WHEN avg_impressions >= 50
             AND avg_clicks < 1
            THEN 'CTR_OPPORTUNITY'

        ELSE 'LOW_VISIBILITY'
    END AS reason_code,

    CASE
        WHEN avg_clicks >= 10
             AND avg_position <= 10
            THEN 'Protect'

        WHEN avg_impressions >= 50
             AND avg_clicks < 1
            THEN 'Improve'

        ELSE 'Monitor'
    END AS action

FROM content_level

ORDER BY baseline_score DESC
"""

In [12]:
baseline = con.sql(baseline_query).df()

print("Unique content rows:", baseline["content_hash_id"].nunique())
print("Total rows:", len(baseline))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique content rows: 155694
Total rows: 155694


In [14]:
from pathlib import Path

Path("work/outputs").mkdir(parents=True, exist_ok=True)

In [15]:
baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

In [16]:
baseline.head(10)

,content_hash_id,avg_impressions,avg_clicks,avg_position,total_impressions,total_clicks,days_observed,baseline_score,reason_code,action
0,content_eadb33b5df496f4a,21280.137931,195.448276,2.383011,617124.0,5668.0,29,1245.893726,HIGH_PERFORMANCE,Protect
1,content_e7b5dd4dff461ad2,6614.354839,78.903226,4.544203,205045.0,2446.0,31,510.027069,HIGH_PERFORMANCE,Protect
2,content_512dbad65bd5ade9,4979.290323,80.838710,3.019798,154358.0,2506.0,31,507.927058,HIGH_PERFORMANCE,Protect
3,content_ec2e0346994fb5a5,8457.793103,51.034483,2.854514,245276.0,1480.0,29,394.186802,HIGH_PERFORMANCE,Protect
4,content_0ec90963d98b97a5,4204.451613,49.225806,3.249691,130338.0,1526.0,31,341.424474,HIGH_PERFORMANCE,Protect
5,content_9a4459a8a3b7a514,1574.277778,44.611111,3.043625,28337.0,803.0,18,292.667460,HIGH_PERFORMANCE,Protect
6,content_f107e54b10b43725,6322.483871,32.129032,3.186054,195997.0,996.0,31,277.311839,HIGH_PERFORMANCE,Protect
7,content_85703b835ab9e744,3898.967742,35.193548,2.728857,120868.0,1091.0,31,269.770848,HIGH_PERFORMANCE,Protect
8,content_7172a7fad43f0998,6640.870968,27.806452,3.367835,205867.0,862.0,31,258.337462,HIGH_PERFORMANCE,Protect
9,content_5fa2737c68998c2e,3517.516129,33.967742,3.796578,109043.0,1053.0,31,256.624136,HIGH_PERFORMANCE,Protect


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
top20 = baseline.head(20)

top20

,content_hash_id,avg_impressions,avg_clicks,avg_position,total_impressions,total_clicks,days_observed,baseline_score,reason_code,action
0,content_eadb33b5df496f4a,21280.137931,195.448276,2.383011,617124.0,5668.0,29,1245.893726,HIGH_PERFORMANCE,Protect
1,content_e7b5dd4dff461ad2,6614.354839,78.903226,4.544203,205045.0,2446.0,31,510.027069,HIGH_PERFORMANCE,Protect
2,content_512dbad65bd5ade9,4979.290323,80.838710,3.019798,154358.0,2506.0,31,507.927058,HIGH_PERFORMANCE,Protect
3,content_ec2e0346994fb5a5,8457.793103,51.034483,2.854514,245276.0,1480.0,29,394.186802,HIGH_PERFORMANCE,Protect
4,content_0ec90963d98b97a5,4204.451613,49.225806,3.249691,130338.0,1526.0,31,341.424474,HIGH_PERFORMANCE,Protect
5,content_9a4459a8a3b7a514,1574.277778,44.611111,3.043625,28337.0,803.0,18,292.667460,HIGH_PERFORMANCE,Protect
6,content_f107e54b10b43725,6322.483871,32.129032,3.186054,195997.0,996.0,31,277.311839,HIGH_PERFORMANCE,Protect
7,content_85703b835ab9e744,3898.967742,35.193548,2.728857,120868.0,1091.0,31,269.770848,HIGH_PERFORMANCE,Protect
8,content_7172a7fad43f0998,6640.870968,27.806452,3.367835,205867.0,862.0,31,258.337462,HIGH_PERFORMANCE,Protect
9,content_5fa2737c68998c2e,3517.516129,33.967742,3.796578,109043.0,1053.0,31,256.624136,HIGH_PERFORMANCE,Protect


In [18]:
top20_review = top20[
    [
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

top20_review

,content_hash_id,baseline_score,reason_code,action
0,content_eadb33b5df496f4a,1245.893726,HIGH_PERFORMANCE,Protect
1,content_e7b5dd4dff461ad2,510.027069,HIGH_PERFORMANCE,Protect
2,content_512dbad65bd5ade9,507.927058,HIGH_PERFORMANCE,Protect
3,content_ec2e0346994fb5a5,394.186802,HIGH_PERFORMANCE,Protect
4,content_0ec90963d98b97a5,341.424474,HIGH_PERFORMANCE,Protect
5,content_9a4459a8a3b7a514,292.667460,HIGH_PERFORMANCE,Protect
6,content_f107e54b10b43725,277.311839,HIGH_PERFORMANCE,Protect
7,content_85703b835ab9e744,269.770848,HIGH_PERFORMANCE,Protect
8,content_7172a7fad43f0998,258.337462,HIGH_PERFORMANCE,Protect
9,content_5fa2737c68998c2e,256.624136,HIGH_PERFORMANCE,Protect


# 3. Top-20 Review

The ranked queue was reviewed by checking the top 20 ranked content items.

### Review Summary

1. Most of the top-ranked items have high impressions, high clicks, and good search positions, so the **Protect** action is reasonable.

2. Some top-ranked rows belong to the same content item on different report dates because the dataset contains one row per content per day.

3. One item (Rank 9) received the **Improve** action even though it has many clicks because its average search position is poor. This shows that the rule gives more importance to search position.

4. Overall, the baseline rule identifies high-performing content, but repeated daily records may affect the ranking.

### What could make the ranking wrong?

- Temporary traffic increases.
- Seasonal changes.
- Daily variations in performance.
- Duplicate daily records for the same content.

# 4. Weak Picks + Leakage Check

### Weak Picks

One weak pick was found during the review.

The content ranked at **Rank 9** was given the **Improve** action because its average search position was greater than 20. However, it also received a high number of clicks. This suggests that the current baseline rule may give too much importance to search position.

In future work, the scoring rule could be improved by balancing clicks, impressions, and search position more effectively.

### Leakage Check

No product flags or future-window information were used in the baseline score.

The score was calculated using only:
- GSC impressions
- GSC clicks
- GSC average position

All of these values are available at the decision time and do not use future information.

Therefore, no data leakage was introduced in this baseline.

# 4. Weak Picks + Leakage Check

### Weak Picks

One weak pick was found during the review.

The content ranked at **Rank 9** was given the **Improve** action because its average search position was greater than 20. However, it also received a high number of clicks. This suggests that the current baseline rule may give too much importance to search position.

In future work, the scoring rule could be improved by balancing clicks, impressions, and search position more effectively.

### Leakage Check

No product flags or future-window information were used in the baseline score.

The score was calculated using only:
- GSC impressions
- GSC clicks
- GSC average position

All of these values are available at the decision time and do not use future information.

Therefore, no data leakage was introduced in this baseline.

# 4. Weak Picks + Leakage Check

### Weak Picks

One weak pick was found during the review.

The content ranked at **Rank 9** was given the **Improve** action because its average search position was greater than 20. However, it also received a high number of clicks. This suggests that the current baseline rule may give too much importance to search position.

In future work, the scoring rule could be improved by balancing clicks, impressions, and search position more effectively.

### Leakage Check

No product flags or future-window information were used in the baseline score.

The score was calculated using only:
- GSC impressions
- GSC clicks
- GSC average position

All of these values are available at the decision time and do not use future information.

Therefore, no data leakage was introduced in this baseline.

✓ Every section above is filled.
✓ The notebook runs top to bottom with no errors.
✓ No client names, URLs, or private queries anywhere.
✓ My claims use careful words.
✓ Committed to my repo.